# Cattle Slaughter & Heat Stress Integration

This notebook integrates USDA weekly cattle slaughter data with MERRA-2 heat stress metrics to analyze relationships between thermal stress and livestock outcomes.

**Analysis Focus:**
- Regions 4 & 6 only (AL, FL, GA, KY, MS, NC, SC, TN, LA, NM, OK, TX)
- Weekly aggregation of daily heat stress metrics
- Time series correlation analysis
- Lag effects (heat stress preceding slaughter)
- Threshold analysis

**Data Sources:**
1. Cattle data: `cattle_data_clean.csv` (weekly, 1984-2025)
2. Heat metrics: `processed_nighttime_recovery/*.nc` (daily, 1984-2025)
3. State mask: `masks/region_mask.nc` (spatial mapping)

In [ ]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
from scipy import stats
from scipy.signal import correlate
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

print("Imports complete!")

In [ ]:
# Configuration - Import from centralized config.py
import sys
from pathlib import Path

# Add project root to path to import config
sys.path.append(str(Path('../..')))  # Up two levels from notebooks/03_analysis/ to research/

# Import paths and constants from config
from config import (
    PROCESSED_NIGHTTIME_DIR,
    PROCESSED_DAYTIME_DIR,
    PROCESSED_VPD_DIR,
    CATTLE_DATA_FILE,
    MASK_FILE,
    FIGURES_DIR,
    STATE_NAMES,
    FOCUS_STATES,
    CATTLE_REGIONS,
    CATTLE_REGION_IDS,
    STATE_REGIONS
)

# Use config paths directly
NIGHTTIME_DIR = PROCESSED_NIGHTTIME_DIR
DAYTIME_DIR = PROCESSED_DAYTIME_DIR
VPD_DIR = PROCESSED_VPD_DIR
CATTLE_FILE = CATTLE_DATA_FILE

# Output directory for this analysis
OUTPUT_DIR = FIGURES_DIR / 'cattle_integration'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project paths loaded from config.py:")
print(f"  Nighttime data: {NIGHTTIME_DIR}")
print(f"  Daytime data: {DAYTIME_DIR}")
print(f"  VPD data: {VPD_DIR}")
print(f"  Cattle data: {CATTLE_FILE}")
print(f"  Mask file: {MASK_FILE}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"\nFocus states: {len(FOCUS_STATES)} states")
print(f"Cattle Region 4: {len(CATTLE_REGIONS['region_4'])} states")
print(f"Cattle Region 6: {len(CATTLE_REGIONS['region_6'])} states")

## 1. Load and Prepare Cattle Data

Load weekly cattle slaughter data for Regions 4 & 6.

In [ ]:
# Load cattle data
df_cattle = pd.read_csv(CATTLE_FILE, parse_dates=['date'])

# Filter for Regions 4 & 6 only
cattle_cols = ['date', 'week', 
               'region_4_dairy', 'region_4_beef_dairy',
               'region_6_dairy', 'region_6_beef_dairy']
df_cattle = df_cattle[cattle_cols].copy()

# Create combined totals
df_cattle['region_4_total'] = df_cattle['region_4_dairy'] + df_cattle['region_4_beef_dairy']
df_cattle['region_6_total'] = df_cattle['region_6_dairy'] + df_cattle['region_6_beef_dairy']
df_cattle['regions_4_6_total'] = df_cattle['region_4_total'] + df_cattle['region_6_total']

# Add year and month for grouping
df_cattle['year'] = df_cattle['date'].dt.year
df_cattle['month'] = df_cattle['date'].dt.month
df_cattle['year_week'] = df_cattle['date'].dt.strftime('%Y-W%U')

print(f"Cattle data shape: {df_cattle.shape}")
print(f"Date range: {df_cattle['date'].min()} to {df_cattle['date'].max()}")
print(f"Total weeks: {len(df_cattle)}")
print(f"\nSummary statistics (1000 head/week):")
print(df_cattle[['region_4_total', 'region_6_total', 'regions_4_6_total']].describe().round(1))

df_cattle.head()

## 2. Load and Process Heat Stress Data

Load daily heat stress metrics and aggregate to weekly averages aligned with cattle data.

In [ ]:
# Helper functions
def load_monthly_data(data_dir, year_start=1984, year_end=2025):
    """Load monthly heat stress data."""
    files = sorted(data_dir.glob('*.nc'))
    datasets = []
    for f in files:
        year_month = f.stem.split('_')[-1]
        year = int(year_month[:4])
        if year_start <= year <= year_end:
            try:
                datasets.append(xr.open_dataset(f))
            except Exception as e:
                print(f"Warning: Could not load {f.name}: {e}")
    if datasets:
        return xr.concat(datasets, dim='time')
    return None

def convert_timedelta_to_hours(data):
    """Convert timedelta64[ns] to float hours if needed."""
    if data.dtype.kind == 'm':  # 'm' = timedelta
        data = data / np.timedelta64(1, 'h')
        data = data.astype(np.float64)
    return data

def compute_regional_mean(data, state_ids, state_mask):
    """Compute spatial mean across multiple states."""
    # Create combined mask for all states in region
    combined_mask = xr.zeros_like(state_mask, dtype=bool)
    for state_id in state_ids:
        combined_mask = combined_mask | (state_mask == state_id)
    
    # Apply mask and compute mean
    masked_data = data.where(combined_mask)
    regional_mean = masked_data.mean(dim=['lat', 'lon'])
    
    # Convert timedelta if needed
    regional_mean = convert_timedelta_to_hours(regional_mean)
    
    return regional_mean

print("Helper functions defined!")

In [ ]:
# Load state mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask
ds_mask.close()

print(f"State mask loaded: {state_mask.shape}")

In [ ]:
# Load nighttime recovery data
print("Loading nighttime recovery data...")
ds_night = load_monthly_data(NIGHTTIME_DIR, 1984, 2025)

if ds_night is not None:
    print(f"Nighttime data loaded: {len(ds_night.time)} days")
    print(f"Date range: {ds_night.time.values[0]} to {ds_night.time.values[-1]}")
    print(f"Variables: {list(ds_night.data_vars)}")
else:
    print("ERROR: Could not load nighttime data")

## 3. Aggregate Heat Metrics to Weekly Resolution

Calculate weekly sums/means of heat stress hours for each region.

In [ ]:
# Define metrics to analyze
metrics = [
    'hours_above_21',
    'hours_above_24',
    'hours_below_18_above_0',
    'hours_below_0'
]

# Map state names to IDs for each cattle region
region_4_ids = [FOCUS_STATES[s] for s in CATTLE_REGIONS['region_4']]
region_6_ids = [FOCUS_STATES[s] for s in CATTLE_REGIONS['region_6'] if s in FOCUS_STATES]

print(f"Region 4 state IDs: {region_4_ids}")
print(f"Region 6 state IDs: {region_6_ids}")

# Compute regional means for each metric
heat_data = {}

for metric in metrics:
    print(f"\nProcessing {metric}...")
    
    # Region 4
    region_4_daily = compute_regional_mean(ds_night[metric], region_4_ids, state_mask)
    
    # Region 6  
    region_6_daily = compute_regional_mean(ds_night[metric], region_6_ids, state_mask)
    
    # Combined regions
    combined_ids = region_4_ids + region_6_ids
    combined_daily = compute_regional_mean(ds_night[metric], combined_ids, state_mask)
    
    # Convert to DataFrame for easier resampling
    df_temp = pd.DataFrame({
        f'region_4_{metric}': region_4_daily.values,
        f'region_6_{metric}': region_6_daily.values,
        f'combined_{metric}': combined_daily.values
    }, index=pd.to_datetime(ds_night.time.values))
    
    # Resample to weekly (sum of hours per week)
    df_weekly = df_temp.resample('W-SAT').sum()  # Weekly ending Saturday to match cattle data
    
    heat_data[metric] = df_weekly
    
    print(f"  Region 4 mean: {region_4_daily.mean().values:.2f} hours/day")
    print(f"  Region 6 mean: {region_6_daily.mean().values:.2f} hours/day")

# Combine all metrics into single DataFrame
df_heat_weekly = pd.concat(heat_data.values(), axis=1)
df_heat_weekly = df_heat_weekly.reset_index().rename(columns={'index': 'date'})

print(f"\nWeekly heat data shape: {df_heat_weekly.shape}")
print(f"Date range: {df_heat_weekly['date'].min()} to {df_heat_weekly['date'].max()}")
df_heat_weekly.head()

## 4. Merge Cattle and Heat Data

Combine weekly cattle slaughter with weekly heat stress metrics.

In [ ]:
# Merge datasets on date
df_merged = pd.merge(df_cattle, df_heat_weekly, on='date', how='inner')

# Remove rows with missing values
df_merged = df_merged.dropna()

print(f"Merged data shape: {df_merged.shape}")
print(f"Date range: {df_merged['date'].min()} to {df_merged['date'].max()}")
print(f"Total weeks: {len(df_merged)}")
print(f"Years covered: {df_merged['year'].nunique()}")

# Save merged dataset (using correct data/ path structure)
output_file = CATTLE_DATA_FILE.parent / 'cattle_heat_merged.csv'
df_merged.to_csv(output_file, index=False)
print(f"\nSaved merged data to: {output_file}")

df_merged.head(10)

## 5. Initial Correlation Analysis

Explore relationships between heat stress metrics and cattle slaughter.

In [ ]:
# Select key variables for correlation
corr_vars = [
    'region_4_total', 'region_6_total', 'regions_4_6_total',
    'region_4_hours_above_21', 'region_4_hours_above_24',
    'region_6_hours_above_21', 'region_6_hours_above_24',
    'combined_hours_above_21', 'combined_hours_above_24'
]

# Compute correlation matrix
corr_matrix = df_merged[corr_vars].corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation: Cattle Slaughter vs Heat Stress (Weekly, 1984-2025)',
            fontweight='bold', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Correlation between cattle slaughter and heat stress:")
print(corr_matrix.loc['regions_4_6_total', ['combined_hours_above_21', 'combined_hours_above_24']])

## 6. Scatter Plots: Heat Stress vs Slaughter

Visualize direct relationships between heat exposure and slaughter rates.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

scatter_pairs = [
    ('region_4_hours_above_24', 'region_4_total', 'Region 4: Hours >24°C vs Slaughter'),
    ('region_6_hours_above_24', 'region_6_total', 'Region 6: Hours >24°C vs Slaughter'),
    ('combined_hours_above_21', 'regions_4_6_total', 'Combined: Hours >21°C vs Slaughter'),
    ('combined_hours_above_24', 'regions_4_6_total', 'Combined: Hours >24°C vs Slaughter')
]

for idx, (heat_var, cattle_var, title) in enumerate(scatter_pairs):
    ax = axes[idx]
    
    # Scatter plot
    scatter = ax.scatter(df_merged[heat_var], df_merged[cattle_var],
                        alpha=0.3, s=20, c=df_merged['month'], cmap='twilight')
    
    # Regression line
    x = df_merged[heat_var].values
    y = df_merged[cattle_var].values
    mask = ~np.isnan(x) & ~np.isnan(y)
    
    if mask.sum() > 10:
        z = np.polyfit(x[mask], y[mask], 1)
        p = np.poly1d(z)
        x_range = np.linspace(x[mask].min(), x[mask].max(), 100)
        ax.plot(x_range, p(x_range), 'r--', linewidth=2, 
               label=f'y = {z[0]:.3f}x + {z[1]:.1f}')
        
        # Compute correlation
        corr, p_val = stats.pearsonr(x[mask], y[mask])
        ax.text(0.05, 0.95, f'r = {corr:.3f}\np < {p_val:.3e}',
               transform=ax.transAxes, va='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_xlabel(heat_var.replace('_', ' ').title() + ' (hours/week)')
    ax.set_ylabel(cattle_var.replace('_', ' ').title() + ' (1000 head/week)')
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    
    # Add colorbar for month
    if idx == 3:
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Month', rotation=270, labelpad=20)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_scatter_heat_vs_slaughter.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Lag Analysis

Investigate if heat stress in previous weeks affects subsequent slaughter rates.

In [ ]:
# Create lagged variables (1-8 weeks)
max_lag = 8
heat_metric = 'combined_hours_above_24'
cattle_metric = 'regions_4_6_total'

lag_correlations = []

for lag in range(0, max_lag + 1):
    # Shift heat data by lag weeks
    df_merged[f'{heat_metric}_lag{lag}'] = df_merged[heat_metric].shift(lag)
    
    # Compute correlation
    corr = df_merged[[cattle_metric, f'{heat_metric}_lag{lag}']].corr().iloc[0, 1]
    lag_correlations.append({'lag': lag, 'correlation': corr})

df_lag = pd.DataFrame(lag_correlations)

# Plot lag correlations
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(df_lag['lag'], df_lag['correlation'], color='steelblue', alpha=0.7, edgecolor='black')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Lag (weeks)', fontsize=12)
ax.set_ylabel('Correlation', fontsize=12)
ax.set_title('Lag Correlation: Heat Stress (Hours >24°C) vs Cattle Slaughter\n(Regions 4 & 6, 1984-2025)',
            fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

# Annotate values
for i, row in df_lag.iterrows():
    ax.text(row['lag'], row['correlation'] + 0.005 * np.sign(row['correlation']),
           f"{row['correlation']:.3f}", ha='center', va='bottom' if row['correlation'] > 0 else 'top',
           fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_lag_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

print("Lag correlations:")
print(df_lag)

## 8. Time Series Visualization

Plot cattle slaughter and heat stress over time to identify patterns.

In [ ]:
# Calculate rolling averages to smooth noise
window = 4  # 4-week moving average

df_merged['cattle_smooth'] = df_merged['regions_4_6_total'].rolling(window=window, center=True).mean()
df_merged['heat_smooth'] = df_merged['combined_hours_above_24'].rolling(window=window, center=True).mean()

# Plot recent 5 years
recent_data = df_merged[df_merged['year'] >= 2019].copy()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Cattle slaughter
ax = axes[0]
ax.plot(recent_data['date'], recent_data['regions_4_6_total'],
       alpha=0.3, color='gray', linewidth=1, label='Weekly')
ax.plot(recent_data['date'], recent_data['cattle_smooth'],
       color='darkblue', linewidth=2, label=f'{window}-week average')
ax.set_ylabel('Cattle Slaughter (1000 head/week)', fontsize=12)
ax.set_title('Cattle Slaughter: Regions 4 & 6 (2019-2025)', fontweight='bold', fontsize=14)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Heat stress
ax = axes[1]
ax.plot(recent_data['date'], recent_data['combined_hours_above_24'],
       alpha=0.3, color='gray', linewidth=1, label='Weekly')
ax.plot(recent_data['date'], recent_data['heat_smooth'],
       color='darkred', linewidth=2, label=f'{window}-week average')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Heat Stress Hours >24°C (hours/week)', fontsize=12)
ax.set_title('Nighttime Heat Stress: Regions 4 & 6 (2019-2025)', fontweight='bold', fontsize=14)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_timeseries_recent.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

This notebook integrated USDA cattle slaughter data with MERRA-2 heat stress metrics:

**Key Findings:**
1. Merged dataset spans 1984-2025 with weekly resolution
2. Correlation analysis reveals relationships between heat exposure and slaughter rates
3. Lag analysis identifies delayed effects (if any)
4. Focus on Regions 4 & 6 (AL, FL, GA, KY, MS, NC, SC, TN, LA, NM, OK, TX)

**Next Steps:**
- Seasonal analysis (see notebook 07)
- Regional comparisons
- Threshold-response relationships
- Extreme event analysis

**Output Files:**
- `cattle_heat_merged.csv` - Integrated weekly dataset
- Correlation matrices
- Scatter plots
- Lag analysis
- Time series visualizations